# Lab: Bifurcation Diagrams

This lab builds the computational tools for analyzing one-dimensional flows. We start with phase portraits, then develop automated bifurcation diagrams, and finish by applying these tools to a classic ecological model — the spruce budworm — where bifurcation theory meets reality.

**Skills developed:**
- Plotting phase portraits with fixed-point classification
- Automated bifurcation diagrams via root-finding
- Identifying saddle-node, transcritical, and pitchfork bifurcations numerically
- Visualizing the cusp catastrophe and hysteresis
- Applying bifurcation analysis to a real ecological model

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../..')
from plot_config import setup_plot_style
setup_plot_style()

## Phase Portraits on the Line

The phase portrait of $\dot{x} = f(x)$ is the most basic tool in dynamical systems. We build a reusable function that takes any $f(x)$, finds its fixed points, classifies their stability, and plots the result.

In [ ]:
def phase_portrait(f, x_range, fp=None, ax=None, title=None):
    """Plot the phase portrait of dx/dt = f(x).
    
    Finds fixed points via sign changes, classifies stability using f'(x*),
    and draws flow arrows between fixed points.
    
    Filled circles = stable, open circles = unstable.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))
    
    x = np.linspace(*x_range, 500)
    y = f(x)
    
    ax.plot(x, y, lw=2)
    ax.axhline(0, color='gray', ls='--', alpha=0.3)
    
    # Find and classify fixed points
    for i in range(len(x) - 1):
        if y[i] * y[i+1] < 0:
            x_star = brentq(f, x[i], x[i+1])
            if fp is not None:
                slope = fp(x_star)
                if slope < 0:
                    ax.plot(x_star, 0, 'o', color='#93C5FD', ms=10, zorder=5)
                else:
                    ax.plot(x_star, 0, 'o', color='#B91C1C', ms=10,
                            mfc='none', mew=2, zorder=5)
            else:
                ax.plot(x_star, 0, 'o', color='#D1B675', ms=10, zorder=5)
    
    # Flow arrows
    arrow_x = np.linspace(x_range[0] + 0.2, x_range[1] - 0.2, 20)
    y_span = max(abs(y.max()), abs(y.min())) or 1
    for xa in arrow_x:
        v = f(xa)
        if abs(v) > 0.05 * y_span:
            ax.annotate('', xy=(xa + 0.12 * np.sign(v), 0),
                        xytext=(xa, 0),
                        arrowprops=dict(arrowstyle='->', color='#D1B675',
                                        lw=1.2, alpha=0.6))
    
    ax.set_xlabel('$x$')
    ax.set_ylabel(r'$\dot{x} = f(x)$')
    if title:
        ax.set_title(title)
    return ax

In [ ]:
# Example: logistic growth and the cubic from Section 3.1
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

r, K = 0.5, 100
phase_portrait(lambda N: r * N * (1 - N / K),
               (- 20, 140),
               fp=lambda N: r * (1 - 2 * N / K),
               ax=axes[0], title=r'Logistic: $\dot{N} = rN(1 - N/K)$')

phase_portrait(lambda x: x - x**3,
               (-1.8, 1.8),
               fp=lambda x: 1 - 3*x**2,
               ax=axes[1], title=r'Cubic: $\dot{x} = x - x^3$')

plt.tight_layout()

## Bifurcation Diagrams

A bifurcation diagram tracks fixed points as a parameter varies. For each value of the parameter $r$, we solve $f(x, r) = 0$ for $x^*$ and classify stability using $f'(x^*, r)$. The result is a plot of $x^*$ versus $r$ with stable branches drawn solid and unstable branches dashed.

We build a general-purpose bifurcation diagram function, then apply it to the three canonical types from Section 3.3.

In [ ]:
def bifurcation_diagram(f, fp, r_range, x_range, n_r=500, ax=None, title=None):
    """Compute and plot a bifurcation diagram.
    
    Parameters
    ----------
    f : callable(x, r), the velocity function
    fp : callable(x, r), the partial derivative df/dx
    r_range : tuple (r_min, r_max)
    x_range : tuple (x_min, x_max), search range for fixed points
    n_r : number of parameter values to scan
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
    
    r_vals = np.linspace(*r_range, n_r)
    x_scan = np.linspace(*x_range, 1000)
    
    stable_r, stable_x = [], []
    unstable_r, unstable_x = [], []
    
    for r in r_vals:
        fx = f(x_scan, r)
        # Find sign changes → fixed points
        for i in range(len(x_scan) - 1):
            if fx[i] * fx[i+1] < 0:
                try:
                    x_star = brentq(lambda x: f(x, r), x_scan[i], x_scan[i+1])
                    slope = fp(x_star, r)
                    if slope < 0:
                        stable_r.append(r); stable_x.append(x_star)
                    else:
                        unstable_r.append(r); unstable_x.append(x_star)
                except ValueError:
                    pass
    
    ax.plot(stable_r, stable_x, '.', color='#93C5FD', ms=1, label='stable')
    ax.plot(unstable_r, unstable_x, '.', color='#B91C1C', ms=1, label='unstable')
    ax.axhline(0, color='gray', ls=':', alpha=0.3)
    ax.axvline(0, color='gray', ls=':', alpha=0.3)
    ax.set_xlabel('$r$')
    ax.set_ylabel('$x^*$')
    if title:
        ax.set_title(title)
    ax.legend(fontsize=9, markerscale=8)
    return ax

In [ ]:
# The three canonical bifurcations, side by side
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

# Saddle-node: dx/dt = r + x^2
bifurcation_diagram(
    f=lambda x, r: r + x**2,
    fp=lambda x, r: 2*x,
    r_range=(-2, 1), x_range=(-2.5, 2.5),
    ax=axes[0], title=r'Saddle-Node: $r + x^2$')

# Transcritical: dx/dt = rx - x^2
bifurcation_diagram(
    f=lambda x, r: r*x - x**2,
    fp=lambda x, r: r - 2*x,
    r_range=(-2, 2), x_range=(-2.5, 2.5),
    ax=axes[1], title=r'Transcritical: $rx - x^2$')

# Supercritical pitchfork: dx/dt = rx - x^3
bifurcation_diagram(
    f=lambda x, r: r*x - x**3,
    fp=lambda x, r: r - 3*x**2,
    r_range=(-2, 2), x_range=(-2.5, 2.5),
    ax=axes[2], title=r'Pitchfork: $rx - x^3$')

plt.suptitle('The Three Canonical Bifurcations', fontsize=13, y=1.02)
plt.tight_layout()

## The Spruce Budworm

The spruce budworm (*Choristoneura fumiferana*) periodically devastates the balsam fir forests of eastern Canada. Ludwig, Jones, and Holling (1978) modeled the budworm population $N$ with a growth term and a predation term:

$$\dot{N} = rN\left(1 - \frac{N}{K}\right) - \frac{BN^2}{A^2 + N^2}$$

The first term is logistic growth. The second is predation by birds, which saturates at high $N$ (the birds can only eat so fast). The predation term has a sigmoidal shape — weak at low $N$, rising steeply near $N = A$, and plateauing at $B$ for large $N$.

The interplay between growth and predation creates a system with **three fixed points** for certain parameter values — a classic setup for saddle-node bifurcations and hysteresis. We non-dimensionalize by setting $x = N/A$ and $\tau = Bt/A$, yielding:

$$\dot{x} = rx\left(1 - \frac{x}{k}\right) - \frac{x^2}{1 + x^2}$$

where $r$ and $k$ are the rescaled growth rate and carrying capacity.

In [ ]:
# Budworm: phase portraits at different carrying capacities
def budworm(x, r, k):
    return r * x * (1 - x / k) - x**2 / (1 + x**2)

def budworm_deriv(x, r, k):
    return r * (1 - 2*x/k) - 2*x / (1 + x**2)**2

r = 0.55
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, k, label in zip(axes,
    [3.0, 7.5, 15.0],
    ['$k = 3$ (refuge only)', '$k = 7.5$ (three equilibria)', '$k = 15$ (outbreak)']):
    phase_portrait(
        lambda x: budworm(x, r, k),
        (0, 18),
        fp=lambda x: budworm_deriv(x, r, k),
        ax=ax, title=label)
    ax.set_ylabel(r'$\dot{x}$')

plt.suptitle(f'Spruce Budworm Phase Portraits ($r = {r}$)', fontsize=13, y=1.02)
plt.tight_layout()

In [ ]:
# Budworm bifurcation diagram: x* vs k (carrying capacity)
fig, ax = plt.subplots(figsize=(8, 5))

r = 0.55
bifurcation_diagram(
    f=lambda x, k: budworm(x, r, k),
    fp=lambda x, k: budworm_deriv(x, r, k),
    r_range=(1, 20), x_range=(0.01, 18),
    n_r=800, ax=ax,
    title=f'Budworm Bifurcation Diagram ($r = {r}$)')

ax.set_xlabel('carrying capacity $k$')
ax.set_ylabel('population $x^*$')
ax.annotate('refuge\n(low pop)', xy=(3, 0.8), fontsize=10, color='#93C5FD')
ax.annotate('outbreak\n(high pop)', xy=(12, 13), fontsize=10, color='#93C5FD')
plt.tight_layout()

The budworm bifurcation diagram shows two saddle-node bifurcations as the carrying capacity $k$ increases. At low $k$, only the "refuge" equilibrium exists (small population, controlled by birds). At high $k$, only the "outbreak" equilibrium exists (large population, overwhelming predation). In between, three equilibria coexist — this is the bistable region where the forest's fate depends on its history.

This is the same cusp structure we saw in Section 3.4, but in a real ecological model. The parameter $k$ controls the forest's capacity to support budworms; as forests grow back after logging, $k$ increases slowly until the system crosses the upper saddle-node and the population erupts.

## The Cusp Surface

Now let's visualize how the imperfect pitchfork $\dot{x} = h + rx - x^3$ behaves across the full $(r, h)$ parameter space.

In [ ]:
# Cusp catastrophe: bifurcation diagrams at several values of h
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)

h_values = [-0.5, -0.2, 0, 0.2, 0.5, 1.0]
for ax, h in zip(axes.flat, h_values):
    bifurcation_diagram(
        f=lambda x, r, _h=h: _h + r*x - x**3,
        fp=lambda x, r, _h=h: r - 3*x**2,
        r_range=(-1, 4), x_range=(-2.5, 2.5),
        n_r=600, ax=ax, title=f'$h = {h}$')

plt.suptitle(r'Slicing the Cusp: $\dot{x} = h + rx - x^3$', fontsize=13, y=1.01)
plt.tight_layout()

## Hysteresis Simulation

Finally, let's simulate what happens when a parameter changes in real time. We integrate $\dot{x} = h + r(t)x - x^3$ where $r(t)$ sweeps slowly from $-1$ to $3$ and back. The system tracks one stable branch until it vanishes, then jumps — and takes a different path on the return.

In [ ]:
# Hysteresis: slowly sweep r(t) forward and backward
h = 0.3
T_half = 500  # time for half-cycle

# Forward sweep: r goes from -1 to 3
def rhs_fwd(t, x):
    r = -1 + 4 * t / T_half
    return [h + r * x[0] - x[0]**3]

sol_fwd = solve_ivp(rhs_fwd, [0, T_half], [-1.2], max_step=1.0, dense_output=True)
t_fwd = np.linspace(0, T_half, 2000)
x_fwd = sol_fwd.sol(t_fwd)[0]
r_fwd = -1 + 4 * t_fwd / T_half

# Backward sweep: r goes from 3 to -1
def rhs_bwd(t, x):
    r = 3 - 4 * t / T_half
    return [h + r * x[0] - x[0]**3]

sol_bwd = solve_ivp(rhs_bwd, [0, T_half], [x_fwd[-1]], max_step=1.0, dense_output=True)
t_bwd = np.linspace(0, T_half, 2000)
x_bwd = sol_bwd.sol(t_bwd)[0]
r_bwd = 3 - 4 * t_bwd / T_half

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: x(t) over the full sweep
axes[0].plot(np.concatenate([t_fwd, t_fwd[-1] + t_bwd]),
             np.concatenate([x_fwd, x_bwd]),
             color='#D1B675', lw=1.5)
axes[0].axvline(T_half, color='gray', ls=':', alpha=0.4)
axes[0].set(xlabel='time $t$', ylabel='$x(t)$',
            title='State over time (forward then backward)')
axes[0].text(T_half * 0.4, -1.4, r'$r$ increasing →', fontsize=10, color='#93C5FD')
axes[0].text(T_half * 1.3, 1.5, r'← $r$ decreasing', fontsize=10, color='#B91C1C')

# Right: hysteresis loop in (r, x) space
axes[1].plot(r_fwd, x_fwd, '-', color='#93C5FD', lw=2, label=r'$r$ increasing')
axes[1].plot(r_bwd, x_bwd, '-', color='#B91C1C', lw=2, label=r'$r$ decreasing')
axes[1].set(xlabel='$r$', ylabel='$x$',
            title=f'Hysteresis Loop ($h = {h}$)')
axes[1].legend(fontsize=9)

plt.tight_layout()